<a href="https://colab.research.google.com/github/RishuKrSingh-coder/DFS-BFS/blob/main/DFS_BFS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CDA305 - Artificial Intelligence Techniques Lab

## Assignment 1: BFS and DFS

### 8-Puzzle Problem

**Name:** Rishu Kumar Singh  
**Roll Number:** 24A12RES541  
**Course:** Artificial Intelligence Techniques Lab  
**Course Code:** CDA305


## 1. Objective

The objective of this assignment is to implement Breadth First Search (BFS)
and Depth First Search (DFS) for solving the 8-puzzle problem.

The 8-puzzle consists of a 3×3 grid containing the numbers 1 to 8 and
one blank space represented by B. In every step, the blank space can move
up, down, left, or right, provided that the movement remains within the grid.

The objective is to determine whether a randomly generated initial state
can reach the fixed target state and compare BFS and DFS based on the
number of steps required and their execution performance.

## 2. Problem Description

The initial state is a randomly generated 3×3 grid containing the numbers
1 to 8 and a blank space B.

The target state is fixed as:

    1 2 3
    4 5 6
    7 8 B

The blank space can move one position at a time in one of four directions:

- Up
- Down
- Left
- Right

The task is to determine whether the target state can be reached from the
randomly generated initial state using these movements.

BFS and DFS are implemented and compared based on the number of moves
required to reach the target.

In [1]:
# Representing the blank space B as 0

start = (
    3, 2, 1,
    4, 5, 6,
    8, 7, 0
)

goal = (
    1, 2, 3,
    4, 5, 6,
    7, 8, 0
)

print("Initial State:")
print(start)

print("\nGoal State:")
print(goal)

Initial State:
(3, 2, 1, 4, 5, 6, 8, 7, 0)

Goal State:
(1, 2, 3, 4, 5, 6, 7, 8, 0)


## 3. State Representation

The blank space B is represented internally by `0`.

For example:

    3 2 1
    4 5 6
    8 7 B

is represented as:

    (3, 2, 1, 4, 5, 6, 8, 7, 0)

A tuple is used because it is immutable and can therefore be stored
inside Python sets for tracking visited states.

In [2]:
def print_puzzle(state):
    for i in range(0, 9, 3):
        row = []

        for value in state[i:i+3]:
            if value == 0:
                row.append("B")
            else:
                row.append(str(value))

        print(" | ".join(row))

    print("-" * 11)

print_puzzle(start)

3 | 2 | 1
4 | 5 | 6
8 | 7 | B
-----------


## 4. Random Initial State Generation

A random permutation of the numbers 1 to 8 and the blank space is
generated to create the initial puzzle state.

The same generated state is used for both BFS and DFS so that the
comparison between the two algorithms is fair.

In [3]:
import random

def generate_random_state():
    numbers = list(range(9))
    random.shuffle(numbers)
    return tuple(numbers)

random_state = generate_random_state()

print("Random Initial State:")
print_puzzle(random_state)

Random Initial State:
6 | 8 | B
5 | 4 | 1
2 | 7 | 3
-----------


## 5. Solvability Check

Not every random 8-puzzle configuration can reach the target state.

To determine whether a configuration is solvable, the number of inversions
is counted while ignoring the blank space.

An inversion occurs when a larger numbered tile appears before a smaller
numbered tile.

For a 3×3 8-puzzle, a state is solvable when the number of inversions is even.

In [4]:
def count_inversions(state):
    values = [x for x in state if x != 0]

    inversions = 0

    for i in range(len(values)):
        for j in range(i + 1, len(values)):
            if values[i] > values[j]:
                inversions += 1

    return inversions


def is_solvable(state):
    return count_inversions(state) % 2 == 0

print("Inversions:", count_inversions(random_state))
print("Solvable:", is_solvable(random_state))

Inversions: 19
Solvable: False


## 6. Generating Valid Moves

The blank space can move in four directions:

- Up
- Down
- Left
- Right

However, a movement is allowed only when the resulting position remains
inside the 3×3 grid.

For every valid movement, a new puzzle state is generated.

In [5]:
def get_neighbors(state):

    neighbors = []

    blank_position = state.index(0)


    row = blank_position // 3
    col = blank_position % 3


    moves = {
        "Up": (-1, 0),
        "Down": (1, 0),
        "Left": (0, -1),
        "Right": (0, 1)
    }


    for move, (row_change, col_change) in moves.items():

        new_row = row + row_change
        new_col = col + col_change

        # Check whether the new position is inside the 3x3 grid
        if 0 <= new_row < 3 and 0 <= new_col < 3:

            new_position = new_row * 3 + new_col


            new_state = list(state)


            new_state[blank_position], new_state[new_position] = \
                new_state[new_position], new_state[blank_position]


            neighbors.append((tuple(new_state), move))

    return neighbors


# Test the function
neighbors = get_neighbors(start)

for state, move in neighbors:
    print("Move:", move)
    print_puzzle(state)

Move: Up
3 | 2 | 1
4 | 5 | B
8 | 7 | 6
-----------
Move: Left
3 | 2 | 1
4 | 5 | 6
8 | B | 7
-----------


## 7. Breadth First Search (BFS)

BFS explores the state space level by level.

It first explores all states that are one move away from the initial state,
then all states that are two moves away, and so on.

Since every movement has the same cost, BFS guarantees the minimum number
of moves required to reach the target, provided that the target is reachable.

In [6]:
from collections import deque

def bfs(start, goal):

    queue = deque([start])

    visited = {start}

    parent = {start: None}

    move_taken = {start: None}

    nodes_expanded = 0

    while queue:

        current = queue.popleft()

        nodes_expanded += 1

        # Goal reached
        if current == goal:
            break

        # Generate neighboring states
        for next_state, move in get_neighbors(current):

            if next_state not in visited:

                visited.add(next_state)

                parent[next_state] = current
                move_taken[next_state] = move

                queue.append(next_state)

    # Goal was not found
    if goal not in parent:
        return None

    return reconstruct_path(
        parent,
        move_taken,
        goal,
        nodes_expanded
    )

## 8. Depth First Search (DFS)

DFS explores one branch of the search space as deeply as possible before
backtracking and exploring another branch.

A stack is used to implement DFS.

Unlike BFS, DFS does not guarantee that the solution found has the minimum
number of moves.

In [7]:
def dfs(start, goal):

    stack = [start]

    visited = {start}

    parent = {start: None}

    move_taken = {start: None}

    nodes_expanded = 0

    while stack:

        current = stack.pop()

        nodes_expanded += 1

        # Goal reached
        if current == goal:
            break

        for next_state, move in get_neighbors(current):

            if next_state not in visited:

                visited.add(next_state)

                parent[next_state] = current
                move_taken[next_state] = move

                stack.append(next_state)

    # Goal was not found
    if goal not in parent:
        return None

    return reconstruct_path(
        parent,
        move_taken,
        goal,
        nodes_expanded
    )

## 9. Solution Path Reconstruction

The parent dictionary stores the previous state from which each state
was reached.

After reaching the target, the path is reconstructed by moving backwards
from the target to the initial state.

In [8]:
def reconstruct_path(parent, move_taken, goal, nodes_expanded):

    states = []
    moves = []

    current = goal

    while current is not None:

        states.append(current)

        if move_taken[current] is not None:
            moves.append(move_taken[current])

        current = parent[current]

    states.reverse()
    moves.reverse()

    return {
        "states": states,
        "moves": moves,
        "steps": len(moves),
        "nodes_expanded": nodes_expanded
    }

## 10. Testing BFS and DFS

Both algorithms are tested using the same initial state and the same
target state.

This ensures that the comparison between BFS and DFS is fair.

In [9]:
# Use a fixed test state first

test_state = (
    3, 2, 1,
    4, 5, 6,
    8, 7, 0
)

print("Initial State:")
print_puzzle(test_state)

print("Goal State:")
print_puzzle(goal)

print("Solvable:", is_solvable(test_state))

Initial State:
3 | 2 | 1
4 | 5 | 6
8 | 7 | B
-----------
Goal State:
1 | 2 | 3
4 | 5 | 6
7 | 8 | B
-----------
Solvable: True


In [10]:
if is_solvable(test_state):

    bfs_result = bfs(test_state, goal)
    dfs_result = dfs(test_state, goal)

    print("BFS Steps:", bfs_result["steps"])
    print("DFS Steps:", dfs_result["steps"])

else:
    print("The puzzle is not solvable.")

BFS Steps: 24
DFS Steps: 40622


In [11]:
if bfs_result is not None:

    print("BFS Solution")
    print("=" * 30)

    for i, state in enumerate(bfs_result["states"]):

        print("Step:", i)
        print_puzzle(state)

    print("Moves:", bfs_result["moves"])
    print("Total Steps:", bfs_result["steps"])

BFS Solution
Step: 0
3 | 2 | 1
4 | 5 | 6
8 | 7 | B
-----------
Step: 1
3 | 2 | 1
4 | 5 | B
8 | 7 | 6
-----------
Step: 2
3 | 2 | B
4 | 5 | 1
8 | 7 | 6
-----------
Step: 3
3 | B | 2
4 | 5 | 1
8 | 7 | 6
-----------
Step: 4
B | 3 | 2
4 | 5 | 1
8 | 7 | 6
-----------
Step: 5
4 | 3 | 2
B | 5 | 1
8 | 7 | 6
-----------
Step: 6
4 | 3 | 2
5 | B | 1
8 | 7 | 6
-----------
Step: 7
4 | 3 | 2
5 | 7 | 1
8 | B | 6
-----------
Step: 8
4 | 3 | 2
5 | 7 | 1
B | 8 | 6
-----------
Step: 9
4 | 3 | 2
B | 7 | 1
5 | 8 | 6
-----------
Step: 10
4 | 3 | 2
7 | B | 1
5 | 8 | 6
-----------
Step: 11
4 | 3 | 2
7 | 1 | B
5 | 8 | 6
-----------
Step: 12
4 | 3 | B
7 | 1 | 2
5 | 8 | 6
-----------
Step: 13
4 | B | 3
7 | 1 | 2
5 | 8 | 6
-----------
Step: 14
4 | 1 | 3
7 | B | 2
5 | 8 | 6
-----------
Step: 15
4 | 1 | 3
7 | 2 | B
5 | 8 | 6
-----------
Step: 16
4 | 1 | 3
7 | 2 | 6
5 | 8 | B
-----------
Step: 17
4 | 1 | 3
7 | 2 | 6
5 | B | 8
-----------
Step: 18
4 | 1 | 3
7 | 2 | 6
B | 5 | 8
-----------
Step: 19
4 | 1 | 3
B | 2 | 6

In [12]:
if dfs_result is not None:

    print("DFS Solution")
    print("=" * 30)

    for i, state in enumerate(dfs_result["states"]):

        print("Step:", i)
        print_puzzle(state)

    print("Moves:", dfs_result["moves"])
    print("Total Steps:", dfs_result["steps"])

Streaming output truncated to the last 5000 lines.
3 | 1 | 8
7 | B | 2
-----------
Step: 39624
4 | 5 | 6
3 | 1 | 8
7 | 2 | B
-----------
Step: 39625
4 | 5 | 6
3 | 1 | B
7 | 2 | 8
-----------
Step: 39626
4 | 5 | 6
3 | B | 1
7 | 2 | 8
-----------
Step: 39627
4 | 5 | 6
B | 3 | 1
7 | 2 | 8
-----------
Step: 39628
4 | 5 | 6
7 | 3 | 1
B | 2 | 8
-----------
Step: 39629
4 | 5 | 6
7 | 3 | 1
2 | B | 8
-----------
Step: 39630
4 | 5 | 6
7 | B | 1
2 | 3 | 8
-----------
Step: 39631
4 | 5 | 6
7 | 1 | B
2 | 3 | 8
-----------
Step: 39632
4 | 5 | B
7 | 1 | 6
2 | 3 | 8
-----------
Step: 39633
4 | B | 5
7 | 1 | 6
2 | 3 | 8
-----------
Step: 39634
B | 4 | 5
7 | 1 | 6
2 | 3 | 8
-----------
Step: 39635
7 | 4 | 5
B | 1 | 6
2 | 3 | 8
-----------
Step: 39636
7 | 4 | 5
1 | B | 6
2 | 3 | 8
-----------
Step: 39637
7 | 4 | 5
1 | 6 | B
2 | 3 | 8
-----------
Step: 39638
7 | 4 | 5
1 | 6 | 8
2 | 3 | B
-----------
Step: 39639
7 | 4 | 5
1 | 6 | 8
2 | B | 3
-----------
Step: 39640
7 | 4 | 5
1 | 6 | 8
B | 2 | 3
-----------

## 11. Execution Time Comparison

The execution time of BFS and DFS is measured using Python's
`time.perf_counter()`.

Both algorithms are given exactly the same initial state and target state.

In [13]:
import time

# BFS timing
start_time = time.perf_counter()

bfs_result = bfs(test_state, goal)

bfs_time = time.perf_counter() - start_time


# DFS timing
start_time = time.perf_counter()

dfs_result = dfs(test_state, goal)

dfs_time = time.perf_counter() - start_time


print("BFS")
print("Steps:", bfs_result["steps"])
print("Nodes Expanded:", bfs_result["nodes_expanded"])
print("Time:", bfs_time, "seconds")

print()

print("DFS")
print("Steps:", dfs_result["steps"])
print("Nodes Expanded:", dfs_result["nodes_expanded"])
print("Time:", dfs_time, "seconds")

BFS
Steps: 24
Nodes Expanded: 122268
Time: 0.5657653380001193 seconds

DFS
Steps: 40622
Nodes Expanded: 44147
Time: 0.22777838499996506 seconds


## 12. Comparison on Multiple Random States

To make the comparison more reliable, multiple random solvable states
are generated.

For every state, both BFS and DFS are executed using the same initial
state and target state.

The following parameters are recorded:

- Number of steps required
- Number of nodes expanded
- Execution time

In [14]:
import pandas as pd
import time

results = []

number_of_tests = 20

test_number = 1

while test_number <= number_of_tests:

    random_state = generate_random_state()

    # Skip unsolvable states
    if not is_solvable(random_state):
        continue

    # BFS
    start_time = time.perf_counter()

    bfs_result = bfs(random_state, goal)

    bfs_time = time.perf_counter() - start_time

    # DFS
    start_time = time.perf_counter()

    dfs_result = dfs(random_state, goal)

    dfs_time = time.perf_counter() - start_time

    results.append({
        "Test": test_number,
        "BFS Steps": bfs_result["steps"],
        "DFS Steps": dfs_result["steps"],
        "BFS Nodes": bfs_result["nodes_expanded"],
        "DFS Nodes": dfs_result["nodes_expanded"],
        "BFS Time": bfs_time,
        "DFS Time": dfs_time
    })

    test_number += 1


results_df = pd.DataFrame(results)

results_df

,Test,BFS Steps,DFS Steps,BFS Nodes,DFS Nodes,BFS Time,DFS Time
0,1,27,54821,173492,63969,0.691623,0.240241
1,2,22,40140,83413,43413,0.274430,0.158380
2,3,20,12392,38658,12746,0.127494,0.057081
3,4,23,56917,117567,67724,0.382751,0.235739
4,5,24,63074,127309,81036,0.424009,0.296775
5,6,22,65942,81600,96943,0.269690,0.337222
6,7,21,53201,66710,130193,0.234955,0.430817
7,8,20,21152,54773,21958,0.173088,0.077956
8,9,20,60554,44644,117611,0.142309,0.413141
9,10,22,17084,80049,17635,0.264644,0.066599


## 13. Results and Analysis

The experimental results show that BFS generally finds solutions using
the minimum number of moves because it explores the state space level by
level.

DFS can sometimes find a solution quickly, but the solution may require
more moves because DFS does not guarantee an optimal path.

The number of nodes expanded and execution time can vary depending on
the initial configuration and the order in which states are explored.

Therefore, execution time alone should not be used to determine which
algorithm is better. The number of solution steps and the search
behavior should also be considered.

## 14. Question 1: BFS vs DFS Based on Number of Steps

### BFS

BFS explores all states at the current depth before moving to the next
depth. Since every movement of the blank space has the same cost, BFS
guarantees the shortest possible solution.

Therefore, if BFS finds a solution requiring 12 moves, there cannot be
another solution requiring fewer than 12 moves.

### DFS

DFS explores one branch deeply before backtracking. Therefore, it can
reach the goal through a longer path even when a shorter path exists.

Hence, DFS does not guarantee the minimum number of steps.

### Comparison

BFS is optimal with respect to the number of moves, whereas DFS may
return a solution with more moves.

## 15. Question 2: Which Algorithm is Faster and When?

There is no single algorithm that is always faster.

BFS can be preferable when the goal is relatively close to the initial
state and when finding the shortest solution is important. BFS explores
states systematically according to their depth.

DFS can sometimes be faster if the goal happens to be located along the
branch that DFS explores first. However, DFS can also spend a significant
amount of time exploring an unproductive branch before reaching the goal.

For the 8-puzzle, BFS has an important advantage because it guarantees
the minimum number of moves. DFS uses less search memory in its basic
form, but its solution is not guaranteed to be optimal.

Therefore:

- BFS is preferable when the shortest solution is required.
- DFS can be useful when memory is limited or when any solution is
  acceptable.
- The actual execution time depends on the initial state and search order.